In [ ]:
from pyspark.sql.functions import *

spark.conf.set(
    "spark.sql.session.timeZone",
    "UTC"
)

catalog = "crypto_lakehouse"
silver_schema = "silver"
gold_schema = "gold"

spark.sql(
    "CREATE SCHEMA IF NOT EXISTS crypto_lakehouse.gold"
)

display(
    spark.sql(
        "SHOW TABLES IN crypto_lakehouse.gold"
    )
)

####Dim assets

In [ ]:
asset_source_df = spark.sql(
    f"""
    SELECT UPPER(asset) AS asset_symbol
    FROM {catalog}.{silver_schema}.binance_account_balances
    WHERE total_balance > 0

    UNION

    SELECT UPPER(asset) AS asset_symbol
    FROM {catalog}.{silver_schema}.binance_funding_account_balances
    WHERE total_balance > 0

    UNION

    SELECT UPPER(asset_spent) AS asset_symbol
    FROM {catalog}.{silver_schema}.binance_trades

    UNION

    SELECT UPPER(asset_received) AS asset_symbol
    FROM {catalog}.{silver_schema}.binance_trades

    UNION

    SELECT UPPER(asset) AS asset_symbol
    FROM {catalog}.{silver_schema}.binance_p2p_transactions

    UNION

    SELECT UPPER(asset) AS asset_symbol
    FROM {catalog}.{silver_schema}.binance_deposits

    UNION

    SELECT UPPER(asset) AS asset_symbol
    FROM {catalog}.{silver_schema}.binance_withdrawals

    UNION

    SELECT UPPER(asset) AS asset_symbol
    FROM {catalog}.{silver_schema}.binance_transfers

    UNION

    SELECT UPPER(asset) AS asset_symbol
    FROM {catalog}.{silver_schema}.binance_price_history

    UNION

    SELECT UPPER(asset) AS asset_symbol
    FROM {catalog}.{silver_schema}.giottus_account_balances
    WHERE total_balance > 0

    UNION

    SELECT UPPER(asset_spent) AS asset_symbol
    FROM {catalog}.{silver_schema}.giottus_trades

    UNION

    SELECT UPPER(asset_received) AS asset_symbol
    FROM {catalog}.{silver_schema}.giottus_trades
    """
)

asset_source_df = (
    asset_source_df
    .filter(col("asset_symbol").isNotNull())
)

print("Useful assets:", asset_source_df.count())

display(asset_source_df.orderBy("asset_symbol"))

In [ ]:
dim_asset_df = (
    asset_source_df
    .select(
        col("asset_symbol")
            .alias("asset_key"),
        col("asset_symbol")
    )
    .withColumn(
        "gold_processed_at",
        current_timestamp()
    )
)

print(
    "Dimension asset rows:",
    dim_asset_df.count()
)

display(
    dim_asset_df.orderBy(
        col("asset_symbol")
    )
)

In [ ]:
(
    dim_asset_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{gold_schema}.dim_asset"
    )
)

display(
    spark.table(
        f"{catalog}.{gold_schema}.dim_asset"
    )
)

#### Dim exchange

In [ ]:
binance_exchanges_df = (
    spark.table(f"{catalog}.{silver_schema}.binance_account_snapshots")
    .select("exchange")
)

giottus_exchanges_df = (
    spark.table(f"{catalog}.{silver_schema}.giottus_account_balances")
    .select("exchange")
)

exchange_source_df = binance_exchanges_df.unionByName(
    giottus_exchanges_df
)

dim_exchange_df = (
    exchange_source_df
    .select(
        upper(col("exchange")).alias("exchange_key"),
        col("exchange").alias("exchange_name")
    )
    .filter(col("exchange_key").isNotNull())
    .dropDuplicates(["exchange_key"])
    .withColumn("exchange_type", lit("Centralized Exchange"))
    .withColumn("gold_processed_at", current_timestamp())
)

display(dim_exchange_df)

In [ ]:
(
    dim_exchange_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{gold_schema}.dim_exchange"
    )
)

display(
    spark.table(
        f"{catalog}.{gold_schema}.dim_exchange"
    )
)


####Dim date

In [ ]:
date_source_df = spark.sql(
    f"""
    SELECT TO_DATE(open_time) AS data_date
    FROM {catalog}.{silver_schema}.binance_price_history

    UNION ALL

    SELECT TO_DATE(trade_time) AS data_date
    FROM {catalog}.{silver_schema}.binance_trades

    UNION ALL

    SELECT TO_DATE(transaction_time) AS data_date
    FROM {catalog}.{silver_schema}.binance_p2p_transactions

    UNION ALL

    SELECT TO_DATE(deposit_time) AS data_date
    FROM {catalog}.{silver_schema}.binance_deposits

    UNION ALL

    SELECT TO_DATE(COALESCE(completed_at, requested_at)) AS data_date
    FROM {catalog}.{silver_schema}.binance_withdrawals

    UNION ALL

    SELECT TO_DATE(transfer_time) AS data_date
    FROM {catalog}.{silver_schema}.binance_transfers

    UNION ALL

    SELECT TO_DATE(SUBSTRING(snapshot_id, 9, 8), 'yyyyMMdd') AS data_date
    FROM {catalog}.{silver_schema}.binance_account_snapshots

    UNION ALL

    SELECT TO_DATE(snapshot_time) AS data_date
    FROM {catalog}.{silver_schema}.binance_funding_account_snapshots

    UNION ALL

    SELECT TO_DATE(trade_time) AS data_date
    FROM {catalog}.{silver_schema}.giottus_trades

    UNION ALL

    SELECT TO_DATE(deposit_time) AS data_date
    FROM {catalog}.{silver_schema}.giottus_fiat_deposits

    UNION ALL

    SELECT TO_DATE(withdrawal_time) AS data_date
    FROM {catalog}.{silver_schema}.giottus_fiat_withdrawals

    UNION ALL

    SELECT TO_DATE(deposit_time) AS data_date
    FROM {catalog}.{silver_schema}.giottus_crypto_deposits

    UNION ALL

    SELECT TO_DATE(withdrawal_time) AS data_date
    FROM {catalog}.{silver_schema}.giottus_crypto_withdrawals

    UNION ALL

    SELECT TO_DATE(bronze_ingested_at) AS data_date
    FROM {catalog}.{silver_schema}.giottus_account_balances
    """
)

date_range_df = (
    date_source_df
    .filter(col("data_date").isNotNull())
    .agg(
        min("data_date").alias("start_date"),
        max("data_date").alias("end_date")
    )
)

display(date_range_df)

In [ ]:
dim_date_df = (
    date_range_df
    .select(
        explode(
            sequence(
                col("start_date"),
                col("end_date")
            )
        ).alias("full_date")
    )
    .select(
        date_format(
            col("full_date"),
            "yyyyMMdd"
        )
        .cast("integer")
        .alias("date_key"),

        col("full_date"),

        year(
            col("full_date")
        ).alias("year"),

        quarter(
            col("full_date")
        ).alias("quarter"),

        month(
            col("full_date")
        ).alias("month_number"),

        date_format(
            col("full_date"),
            "MMMM"
        ).alias("month_name"),

        dayofmonth(
            col("full_date")
        ).alias("day_of_month"),

        date_format(
            col("full_date"),
            "EEEE"
        ).alias("day_name"),

        weekofyear(
            col("full_date")
        ).alias("week_of_year"),

        when(
            dayofweek(
                col("full_date")
            ).isin(1, 7),
            True
        )
        .otherwise(False)
        .alias("is_weekend")
    )
    .withColumn(
        "gold_processed_at",
        current_timestamp()
    )
)


print(
    "Date dimension rows:",
    dim_date_df.count()
)


display(
    dim_date_df.orderBy(
        col("full_date")
    )
)

In [ ]:
(
    dim_date_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{gold_schema}.dim_date"
    )
)


saved_dim_date_df = spark.table(
    f"{catalog}.{gold_schema}.dim_date"
)


print(
    "Saved date rows:",
    saved_dim_date_df.count()
)


display(
    saved_dim_date_df
    .agg(
        min("full_date").alias("start_date"),
        max("full_date").alias("end_date")
    )
)

####Dim wallet

In [ ]:
from pyspark.sql.functions import col, current_timestamp

wallet_rows = [
    (
        "BINANCE_SPOT",
        "Binance Spot Wallet",
        "SPOT",
        "BINANCE",
        "CUSTODIAL",
        None,
        True
    ),
    (
        "BINANCE_FUNDING",
        "Binance Funding Wallet",
        "FUNDING",
        "BINANCE",
        "CUSTODIAL",
        None,
        True
    ),
    (
        "GIOTTUS_SPOT",
        "Giottus Spot Wallet",
        "SPOT",
        "GIOTTUS",
        "CUSTODIAL",
        None,
        True
    ),
    (
        "PHANTOM_SOLANA",
        "Phantom Solana Wallet",
        "HOT_WALLET",
        "PHANTOM",
        "SELF_CUSTODY",
        "SOLANA",
        True
    )
]

wallet_columns = [
    "wallet_key",
    "wallet_name",
    "wallet_type",
    "provider",
    "custody_type",
    "network",
    "is_active"
]

dim_wallet_df = (
    spark.createDataFrame(wallet_rows, wallet_columns)
    .withColumn("gold_processed_at", current_timestamp())
)

print("Wallet rows:", dim_wallet_df.count())
display(dim_wallet_df.orderBy("wallet_key"))

In [ ]:
(
    dim_wallet_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{gold_schema}.dim_wallet"
    )
)